# Data Exploration

This notebook explores the **SOOP (Stroke Outcome Optimization Project)** dataset used for
stroke detection with Graph Attention Networks on brain MRI.

## Dataset Overview

- **Subjects:** 1,715 patients with acute ischemic stroke
- **Modalities:** T1-weighted, FLAIR, ADC (Apparent Diffusion Coefficient), TRACE (DWI b=1000)
- **Labels:** 3-class voxel-level segmentation
  - 0 = Normal tissue
  - 1 = Ischemic penumbra (at-risk tissue)
  - 2 = Infarct core (irreversibly damaged)
- **Atlas:** Harvard-Oxford cortical/subcortical atlas (96 regions) registered to each subject
- **Resolution:** 1mm isotropic, skull-stripped and co-registered

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from stroke_gat.config import Config
from stroke_gat.data.service import DataService

print("Imports successful.")

In [ ]:
# Load project configuration
cfg = Config.from_yaml("../configs/default.yaml")
print(f"Data root: {cfg.data.root_dir}")
print(f"Modalities: {cfg.data.modalities}")
print(f"Number of classes: {cfg.data.num_classes}")

# Create the DataService which handles all I/O
data_service = DataService(cfg)

In [ ]:
# Discover all available subjects in the dataset
subjects = data_service.discover_subjects()
print(f"Total subjects found: {len(subjects)}")
print(f"First 10 subject IDs: {subjects[:10]}")
print(f"Last 5 subject IDs:  {subjects[-5:]}")

In [ ]:
# Load one subject's multimodal MRI volumes
subject_id = subjects[0]
print(f"Loading subject: {subject_id}")

volumes = data_service.load_subject(subject_id)

print("\nLoaded modalities and shapes:")
for modality, vol in volumes.items():
    print(f"  {modality:>8s}: shape={vol.shape}, dtype={vol.dtype}, "
          f"range=[{vol.min():.3f}, {vol.max():.3f}]")

In [ ]:
# Plot all 4 modalities side by side at the middle axial slice
modality_names = ["T1", "FLAIR", "ADC", "TRACE"]
mid_slice = volumes["T1"].shape[2] // 2  # middle axial slice

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, modality_names):
    vol = volumes[name]
    ax.imshow(vol[:, :, mid_slice].T, cmap="gray", origin="lower")
    ax.set_title(name, fontsize=14, fontweight="bold")
    ax.axis("off")

fig.suptitle(f"Subject {subject_id} - Axial Slice {mid_slice}",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Load the atlas parcellation and visualize overlay on T1
atlas = data_service.load_atlas(subject_id)
print(f"Atlas shape: {atlas.shape}")
print(f"Number of unique regions: {len(np.unique(atlas)) - 1}")  # exclude background

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# T1 alone
axes[0].imshow(volumes["T1"][:, :, mid_slice].T, cmap="gray", origin="lower")
axes[0].set_title("T1-weighted", fontsize=13)
axes[0].axis("off")

# T1 with atlas overlay
axes[1].imshow(volumes["T1"][:, :, mid_slice].T, cmap="gray", origin="lower")
atlas_slice = atlas[:, :, mid_slice].T.astype(float)
atlas_slice[atlas_slice == 0] = np.nan  # make background transparent
axes[1].imshow(atlas_slice, cmap="nipy_spectral", alpha=0.4, origin="lower")
axes[1].set_title("T1 + Atlas Overlay", fontsize=13)
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Load ground-truth lesion masks and overlay on FLAIR
mask = data_service.load_mask(subject_id)
print(f"Mask shape: {mask.shape}")
print(f"Label distribution:")
for label, name in enumerate(["Normal", "Penumbra", "Core"]):
    count = np.sum(mask == label)
    pct = 100.0 * count / mask.size
    print(f"  {name:>10s} (label={label}): {count:>10,} voxels ({pct:.2f}%)")

# Create custom colormap for lesion classes
lesion_cmap = ListedColormap(["none", "yellow", "red"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# FLAIR alone
axes[0].imshow(volumes["FLAIR"][:, :, mid_slice].T, cmap="gray", origin="lower")
axes[0].set_title("FLAIR", fontsize=13)
axes[0].axis("off")

# Lesion mask alone
axes[1].imshow(mask[:, :, mid_slice].T, cmap=lesion_cmap,
               vmin=0, vmax=2, origin="lower")
axes[1].set_title("Lesion Mask", fontsize=13)
axes[1].axis("off")

# FLAIR + lesion overlay
axes[2].imshow(volumes["FLAIR"][:, :, mid_slice].T, cmap="gray", origin="lower")
mask_overlay = mask[:, :, mid_slice].T.astype(float)
mask_overlay[mask_overlay == 0] = np.nan
axes[2].imshow(mask_overlay, cmap=lesion_cmap, alpha=0.5,
               vmin=0, vmax=2, origin="lower")
axes[2].set_title("FLAIR + Lesion Overlay", fontsize=13)
axes[2].axis("off")

fig.suptitle(f"Subject {subject_id} - Lesion Segmentation",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

Key observations from the SOOP dataset:

1. **Multimodal complementarity:** Each MRI modality captures different tissue properties:
   - **T1:** Anatomical structure with good gray/white matter contrast
   - **FLAIR:** Highlights edema and chronic white matter changes
   - **ADC:** Quantifies water diffusion; restricted diffusion in acute infarct appears dark
   - **TRACE (DWI):** High signal in acute ischemic regions

2. **Class imbalance:** Lesion voxels (penumbra + core) constitute a very small fraction
   of total brain volume, motivating the use of graph-based approaches that can
   aggregate information at the supervoxel level.

3. **Atlas registration:** The Harvard-Oxford atlas provides anatomical context for each
   subject, constraining the SLIC supervoxel segmentation to respect anatomical boundaries.

Next: See `02_parcellation_and_slic_demo.ipynb` for how we generate supervoxels.